# Tome 12, Chapter 6: Local fields (Python companion)

Exercises first, solutions at the end. We compute $p$-adic roots by Newton's method, factor polynomials over $\mathbb Q_p$ to read off the completions of a number field (Theorem 6.18), count all extensions of $\mathbb Q_p$ of small degree and check Serre's mass formula, watch Krasner's lemma at work, and end with the computer-science side of Hensel's lemma: Newton inversion modulo $2^{64}$ and Hensel lifting of factorisations, the engine of polynomial factoring over $\mathbb Z$. Requires `cypari2`.

In [ ]:
import math, random, time
from fractions import Fraction
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)

## Exercises

**P6.1 (★) Newton doubles the digits.** Compute $\sqrt2\in\mathbb Z_7$ modulo $7^{64}$ by Newton's iteration $x\mapsto x-\frac{x^2-2}{2x}$ with exact integers, printing the number of correct digits after each step. Compare with PARI's `sqrt(2+O(7^64))`.

**P6.2 (★) Completions from a factorisation.** For $f=X^3-2$ and $f=X^5-X-1$ and the primes $p<30$, factor $f$ over $\mathbb Q_p$ (`factorpadic`) and compare the degrees of the factors with the $e_if_i$ of the primes above $p$ in $\mathbb Q(\theta)$ (`idealprimedec`), as Theorem 6.18 predicts.

**P6.3 (★★) How many extensions?** `padicfields(p, n, 1)` lists the extensions of $\mathbb Q_p$ of degree $n$ up to isomorphism, with $e$, $f$, the valuation $s$ of the discriminant, and the number $c$ of distinct conjugate fields inside $\overline{\mathbb Q}_p$. Count the fields of degree $n\le6$ for $p=2,3,5$ (up to isomorphism and inside $\overline{\mathbb Q}_p$), and check Serre's mass formula $\sum_L p^{-(s_L-n+1)}=n$ over the totally ramified ones.

**P6.4 (★★) Krasner's lemma, sharp.** Let $g=X^2-2$ over $\mathbb Q_2$, with roots $\pm\sqrt2$, and $h_t=X^2-(2+2^kt)$ for odd $t$. For which $k$ is $\mathbb Q_2(\sqrt{2+2^kt})=\mathbb Q_2(\sqrt2)$? Compare with the bound $|h-g|<\delta^2$ of Proposition 6.14.

**P6.5 (★★★) CS application: Hensel lifting in computer algebra.**
1. *Newton inversion.* Montgomery multiplication needs $a^{-1}\bmod 2^{64}$ for odd $a$. Show that $x\mapsto x(2-ax)$ doubles the number of correct bits, that $x_0=a$ is already correct modulo $8$, and that four steps suffice for $64$ bits. Implement and test it.
2. *Lifting a factorisation.* Implement the lifting of Theorem 6.6: given $f\in\mathbb Z[X]$ monic and a coprime factorisation $\bar f=\bar g\bar h$ modulo $p$, lift it modulo $p^k$. Use it to factor $f=(X^3-5X+11)(X^2+X+7)$ (expanded) over $\mathbb Z$ from its factorisation modulo a suitable small prime, by lifting to $p^k>2B$ where $B$ bounds the coefficients of the factors, and taking symmetric residues. Why does a single prime not always suffice (think of $X^4+1$)?

## Solutions

### P6.1
**How to find it.** If $x\equiv\sqrt2\pmod{7^m}$, then $x-\frac{x^2-2}{2x}\equiv\sqrt2\pmod{7^{2m}}$; work modulo $7^{2m}$ at each step, with `pow(2*x, -1, M)` for the inverse.

In [ ]:
p, N = 7, 64
x, m = 3, 1
while m < N:
    m = min(2 * m, N)
    M = p**m
    x = (x - (x * x - 2) * pow(2 * x, -1, M)) % M
    print(f"{m:3d} digits correct: (x^2 - 2) divisible by 7^{m}: {(x*x - 2) % M == 0}")
def digits(x, p, n):
    out = []
    for _ in range(n):
        out.append(x % p); x //= p
    return out
print("our digits :", digits(x, 7, 16))
y = pari(f"sqrt(2+O(7^{N}))")
print("PARI       :", digits(int(pari.truncate(y)) % 7**N, 7, 16))

Six Newton steps give $64$ digits, against $64$ steps for the digit-by-digit method of Tome 1. Same phenomenon as in $\mathbb R$: quadratic convergence. Our root and PARI's agree; PARI's convention picks the root $\equiv3\pmod7$ here.

### P6.2
**How to find it.** `factorpadic(f, p, r)` factors over $\mathbb Q_p$ to precision $p^r$; the degree of each factor is $[K_P:\mathbb Q_p]=e_Pf_P$.

In [ ]:
for fs in ("x^3-2", "x^5-x-1"):
    f = pari(fs); nf = pari.nfinit(f)
    ok = True; rows = []
    for p in pari.primes([2, 30]):
        p = int(p)
        local = sorted(int(pari.poldegree(g)) for g in pari.factorpadic(f, p, 10)[0])
        glob = sorted(int(P[2]) * int(P[3]) for P in pari.idealprimedec(nf, p))
        ok &= (local == glob)
        rows.append(f"{p}:{local}")
    print(fs, "local degrees = e*f for every p:", ok)
    print("   ", " ".join(rows))

For $X^3-2$: $p=2,3$ are totally ramified (one factor of degree $3$); $p\equiv2\pmod3$ gives degrees $[1,2]$; $p\equiv1\pmod 3$ gives $[1,1,1]$ or $[3]$ according to whether $2$ is a cube modulo $p$, exactly as in Chapter 2. For $X^5-X-1$ (discriminant $2869=19\cdot151$), the prime $19$ gives a degree-$2$ ramified factor next to unramified ones.

### P6.3
**How to find it.** Each isomorphism class of degree $n$ appears $c$ times inside $\overline{\mathbb Q}_p$ (its conjugate fields). In the mass formula, sum $c\,p^{-(s-n+1)}$ over the classes with $e=n$.

In [ ]:
for p in (2, 3, 5):
    for n in range(2, 7):
        L = pari.padicfields(p, n, 1)
        classes = len(L)
        fields = sum(int(c) for (P, e, f, s, c) in L)
        mass = sum(Fraction(int(c), p**(int(s) - n + 1)) for (P, e, f, s, c) in L if int(e) == n)
        print(f"p={p} n={n}: {classes:4d} classes, {fields:5d} fields in Qp-bar, mass of totally ramified = {mass}")

The mass is exactly $n$ every time. Note the jump when $p\mid n$ ("wild" ramification): $\mathbb Q_2$ has $59$ quartic extensions up to isomorphism, $\mathbb Q_3$ has $10$ cubic ones, while for $p\nmid n$ the counts are small, because tame extensions are all of the form of Exercise 6.6.

### P6.4
**How to find it.** $\mathbb Q_2(\sqrt a)=\mathbb Q_2(\sqrt b)$ iff $a/b$ is a square, and a unit $u$ of $\mathbb Z_2$ is a square iff $u\equiv1\pmod 8$ (Tome 1, Corollary 12.8). Here $a/b=1+2^{k-1}t$.

In [ ]:
for k in range(2, 8):
    same = [bool(pari(f"issquare((2+2^{k}*{t})/2 + O(2^30))")) for t in (1, 3, 5, 7)]
    print(f"k={k}: same field for odd t = 1,3,5,7: {same}")
print("Krasner: need |2^k t| < delta^2 = |2 sqrt2|^2 = 2^-3, i.e. k >= 4")

The fields agree exactly for $k\ge4$: $1+2^{k-1}t\equiv1\pmod8$ iff $k\ge4$. Krasner's bound $|h-g|<\delta^e=|2\sqrt2|^2=2^{-3}$ says the same thing, $k\ge4$: in this example the lemma is sharp. For $k=3$ the field changes ($\mathbb Q_2(\sqrt{10})$ instead of $\mathbb Q_2(\sqrt2)$), and for $k=2$ we get $\mathbb Q_2(\sqrt{6})$ or $\mathbb Q_2(\sqrt{-2})$.

### P6.5
**How to find it.** (1) If $ax\equiv1-\varepsilon$, then $a\cdot x(2-ax)=(1-\varepsilon)(1+\varepsilon)=1-\varepsilon^2$: the error squares. Every odd $a$ satisfies $a^2\equiv1\pmod8$, so $x_0=a$ has $3$ correct bits, and $3\to6\to12\to24\to48\to96$: five steps for $64$ bits, four if we start from a $4$-bit table (or from $x_0=(3a)\oplus2$, which is correct to $5$ bits). (2) Follow the proof of Theorem 6.6 literally, with polynomial arithmetic modulo $p$ for the Bezout step.

In [ ]:
MASK = (1 << 64) - 1
def inv64(a):
    x = a                              # correct mod 2^3
    for _ in range(5):
        x = (x * (2 - a * x)) & MASK
    return x
random.seed(0)
print(all((a * inv64(a)) & MASK == 1 for a in (random.getrandbits(64) | 1 for _ in range(10000))))
a = 0x9E3779B97F4A7C15
x = a; bits = []
for _ in range(5):
    x = (x * (2 - a * x)) & MASK
    err = (a * x - 1) & MASK
    bits.append(64 if err == 0 else (err & -err).bit_length() - 1)
print("correct bits after each step:", bits)

In [ ]:
# Polynomials as lists of ints, constant term first.
def trim(f):
    while len(f) > 1 and f[-1] == 0:
        f = f[:-1]
    return f
def add(f, g, m):
    n = max(len(f), len(g))
    return trim([((f[i] if i < len(f) else 0) + (g[i] if i < len(g) else 0)) % m for i in range(n)])
def sub(f, g, m):
    return add(f, [-c for c in g], m)
def mul(f, g, m):
    h = [0] * (len(f) + len(g) - 1)
    for i, a in enumerate(f):
        if a:
            for j, b in enumerate(g):
                h[i + j] = (h[i + j] + a * b) % m
    return trim(h)
def divmod_p(f, g, p):
    # division in F_p[X] (g with invertible leading coefficient)
    f = [c % p for c in f]; q = [0] * max(1, len(f) - len(g) + 1)
    inv = pow(g[-1], -1, p)
    while len(f) >= len(g) and any(f):
        c = f[-1] * inv % p; d = len(f) - len(g)
        q[d] = c
        f = sub(f, [0] * d + [c * x for x in g], p)
        if len(f) < len(g) or f == [0]:
            break
    return trim(q), trim(f)
def xgcd_p(f, g, p):
    # returns a, b with a f + b g = 1 in F_p[X] (f, g coprime)
    r0, r1, a0, a1, b0, b1 = f, g, [1], [0], [0], [1]
    while trim(r1) != [0]:
        q, r = divmod_p(r0, r1, p)
        r0, r1 = r1, r
        a0, a1 = a1, sub(a0, mul(q, a1, p), p)
        b0, b1 = b1, sub(b0, mul(q, b1, p), p)
    c = pow(r0[0], -1, p)            # r0 is a non-zero constant
    return [x * c % p for x in a0], [x * c % p for x in b0]
def hensel_lift(f, g0, h0, p, k):
    # f = g h mod p^(j+1) for j = 0..k-1, g monic (Theorem 6.6, linear lifting)
    a, b = xgcd_p(g0, h0, p)
    g, h = g0[:], h0[:]
    for j in range(1, k):
        m = p**(j + 1)
        c = [x // p**j for x in sub(f, mul(g, h, m), m)]   # f - g h = p^j c  (mod p^(j+1))
        q, v = divmod_p(mul(b, c, p), g0, p)
        u = add(mul(a, c, p), mul(q, h0, p), p)
        g = add(g, [p**j * x for x in v], m)
        h = add(h, [p**j * x for x in u], m)
    return g, h
def sym(f, m):
    return [x - m if x > m // 2 else x for x in f]

G = [11, -5, 0, 1]          # X^3 - 5X + 11
H = [7, 1, 1]               # X^2 + X + 7
def mulZ(f, g):
    h = [0] * (len(f) + len(g) - 1)
    for i, a in enumerate(f):
        for j, b in enumerate(g):
            h[i + j] += a * b
    return h
f = mulZ(G, H)
print("f =", f)
for p in pari.primes([3, 40]):
    p = int(p)
    fac = pari.factormod(pari(f"Pol({list(reversed(f))})"), p)
    degs = [int(pari.poldegree(g)) for g in fac[0]]
    if sorted(degs) == [2, 3] and all(int(e) == 1 for e in fac[1]):
        break
print("p =", p, "factors mod p of degrees", degs)
g0 = [int(pari.lift(c)) for c in reversed(pari.Vec(pari.lift(fac[0][degs.index(3)])))]
h0 = [int(pari.lift(c)) for c in reversed(pari.Vec(pari.lift(fac[0][degs.index(2)])))]
k = 1
while p**k <= 2 * 200:
    k += 1
g, h = hensel_lift(f, g0, h0, p, k)
print("lifted mod", p, "^", k, ":", sym(g, p**k), sym(h, p**k))
print("product equals f over Z:", mulZ(sym(g, p**k), sym(h, p**k)) == f)

The cubic and quadratic factors come back exactly, with coefficients read as symmetric residues modulo $p^k$. This is the Berlekamp–Zassenhaus algorithm (1969) in miniature: factor modulo $p$ (Cantor–Zassenhaus, Chapter 2's notebook), lift with Hensel, then try products of lifted factors. The last step can be expensive: $X^4+1$ is irreducible over $\mathbb Z$ but splits modulo every prime into factors of degree $\le2$ (Exercise 2.3, $\mathbb Q(\zeta_8)$ has no cyclic decomposition group of order $4$), so the lifted factors must be recombined, in the worst case over exponentially many subsets. Van Hoeij's algorithm (2002) solves the recombination with lattice reduction, the LLL of Chapter 3's notebook, and makes factoring over $\mathbb Z$ fast in practice.